# 1. Import necessary packages

In [1]:
import os

import turmoric
from turmoric.image_process import nd2_to_tif
from turmoric.utils import recursively_get_all_filepaths
from turmoric.apply_thresholds import apply_all_thresh

from pathlib import Path

import pandas as pd
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
from pathlib import Path
from skimage import filters, io
from skimage.segmentation import clear_border
from skimage.morphology import remove_small_objects
from skimage.measure import block_reduce, label, regionprops, regionprops_table
from skimage import exposure
from scipy import ndimage

from turmoric.cell_analysis import apply_regionprops_recursively
from turmoric.utils import organize_files_without_leakage
from turmoric.vampire_model import VampireModelTrainer
from turmoric.split_functions import *
from turmoric.grouping_strategy import CustomFunctionStrategy


from PIL import Image

from sklearn.model_selection import train_test_split

import sklearn
import shutil
from typing import List, Dict, Callable, Optional, Any


import vampire
from vampire.plot import *

import matplotlib.pyplot as plt

import warnings
import logging

# Suppress matplotlib font warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)


import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold

import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import StandardScaler
from pathlib import Path

In [2]:
all_tifs = recursively_get_all_filepaths("/gscratch/cheme/nlsschim/data/cross_species/tifs/all_data/ferret/thresholded", file_type=".tif")
print(len(all_tifs))

138


In [5]:
props_list = ('area', 'bbox_area', 'centroid', 'convex_area',
              'eccentricity', 'equivalent_diameter', 'euler_number',
              'extent', 'filled_area', 'major_axis_length',
              'minor_axis_length', 'orientation', 'perimeter', 'solidity',
              'feret_diameter_max')

In [3]:
all_tifs[0]

'/gscratch/cheme/nlsschim/data/cross_species/tifs/all_data/ferret/thresholded/20230308_ferret_086L_60x_denoised_pl1_Alexa_488_antibody_otsu_thresh.tif'

In [7]:
regionprops_dfs = []

for file in all_tifs:
    im = tiff.imread(file)

    label_im = label(im)
    props = regionprops_table(label_im, properties=props_list)

    props_df = pd.DataFrame(props)
    props_df['filename'] = file
    

    filename = file.split("/")[10]
    print(filename)

    props_df['species'] = "ferret"#filename.split("_")[1]
    props_df['sex'] = "female"
    props_df["age"] = "p1"
    props_df["region"] = filename.split("_")[-4][:-1]
    props_df["animal_id"] = filename.split("_")[2]
    regionprops_dfs.append(props_df)

all_regionprops_data = pd.concat(regionprops_dfs)

20230308_ferret_086L_60x_denoised_pl1_Alexa_488_antibody_otsu_thresh.tif
20230306_ferret_086j_60x_denoised_thal2_Alexa_488_antibody_otsu_thresh.tif
20230308_ferret_085B_60x_denoised_cbl3_Alexa_488_antibody_otsu_thresh.tif
20230309_ferret_085A_60x_denoised_cc2_Alexa_488_antibody_otsu_thresh.tif
20230305_ferret_086h_60x_denoised_WMcor1_Alexa_488_antibody_otsu_thresh.tif
20230310_ferret_0859_60x_denoised_thal3_Alexa_488_antibody_otsu_thresh.tif
20230310_ferret_0858_60x_denoised_WMcor2_Alexa_488_antibody_otsu_thresh.tif
20230310_ferret_0858_60x_denoised_thal1_Alexa_488_antibody_otsu_thresh.tif
20230309_ferret_085C_60x_denoised_st1_Alexa_488_antibody_otsu_thresh.tif
20230310_ferret_0858_60x_denoised_WMcor1_Alexa_488_antibody_otsu_thresh.tif
20230310_ferret_0859_60x_denoised_cc2_Alexa_488_antibody_otsu_thresh.tif
20230308_ferret_085B_60x_denoised_WMcor3_Alexa_488_antibody_otsu_thresh.tif
20230309_ferret_085C_60x_denoised_pl1_Alexa_488_antibody_otsu_thresh.tif
20230309_ferret_085A_60x_denoise

In [8]:
all_regionprops_data["animal_id"].unique()

<StringArray>
['086L', '086j', '085B', '085A', '086h', '0859', '0858', '085C', '086k',
 '086i', '085D', '086M']
Length: 12, dtype: str

In [89]:
# regionprops_df = apply_regionprops_recursively(input_folder="/gscratch/cheme/nlsschim/data/cross_species/tifs/all_data/", 
#                                                properties_list=props_list)
all_regionprops_data['circularity'] = 4*np.pi*all_regionprops_data.area/all_regionprops_data.perimeter**2
all_regionprops_data['aspect_ratio'] = all_regionprops_data.major_axis_length/all_regionprops_data.minor_axis_length
all_regionprops_data

,area,bbox_area,centroid-0,centroid-1,convex_area,eccentricity,equivalent_diameter,euler_number,extent,filled_area,...,perimeter,solidity,feret_diameter_max,filename,species,sex,age,region,circularity,aspect_ratio
0,982.0,2820.0,55.524440,311.291242,1734.0,0.929328,35.359882,1,0.348227,982.0,...,248.793939,0.566321,67.119297,/gscratch/cheme/nlsschim/data/cross_species/ti...,pig,female,p1,cc,0.199362,2.708162
1,428.0,676.0,49.892523,167.887850,470.0,0.665666,23.344090,1,0.633136,428.0,...,87.012193,0.910638,29.154759,/gscratch/cheme/nlsschim/data/cross_species/ti...,pig,female,p1,cc,0.710384,1.340034
2,2378.0,7920.0,151.554247,268.550042,3486.0,0.977343,55.025118,1,0.300253,2378.0,...,397.268073,0.682157,122.967475,/gscratch/cheme/nlsschim/data/cross_species/ti...,pig,female,p1,cc,0.189345,4.724552
3,233.0,361.0,127.914163,392.931330,250.0,0.598853,17.223960,1,0.645429,233.0,...,59.112698,0.932000,21.023796,/gscratch/cheme/nlsschim/data/cross_species/ti...,pig,female,p1,cc,0.837923,1.248659
4,852.0,2914.0,150.550469,149.538732,1470.0,0.962908,32.936304,1,0.292382,852.0,...,241.965512,0.579592,71.610055,/gscratch/cheme/nlsschim/data/cross_species/ti...,pig,female,p1,cc,0.182870,3.706051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38,179.0,575.0,430.782123,291.217877,347.0,0.337822,15.096684,1,0.311304,179.0,...,97.254834,0.515850,25.317978,/gscratch/cheme/nlsschim/data/cross_species/ti...,pig,female,p1,cc,0.237816,1.062462
39,150.0,270.0,449.620000,318.486667,183.0,0.843335,13.819766,1,0.555556,150.0,...,57.455844,0.819672,19.416488,/gscratch/cheme/nlsschim/data/cross_species/ti...,pig,female,p1,cc,0.570996,1.860853
40,204.0,817.0,483.333333,324.985294,450.0,0.881254,16.116478,1,0.249694,204.0,...,105.805087,0.453333,43.566042,/gscratch/cheme/nlsschim/data/cross_species/ti...,pig,female,p1,cc,0.228995,2.115764
41,230.0,782.0,478.630435,379.760870,454.0,0.910548,17.112717,1,0.294118,230.0,...,128.982756,0.506608,35.171011,/gscratch/cheme/nlsschim/data/cross_species/ti...,pig,female,p1,cc,0.173730,2.418940


In [81]:
medians = all_regionprops_data[["area", "bbox_area", "convex_area", "perimeter", "extent", "solidity", "circularity", "aspect_ratio", "eccentricity"]].median()
medians

area             235.500000
bbox_area       1138.000000
convex_area      630.500000
perimeter        133.012193
extent             0.218070
solidity           0.378505
circularity        0.151768
aspect_ratio       2.245934
eccentricity       0.895407
dtype: float64

In [90]:
all_regionprops_data.to_csv("pig_p1.csv", index=False)

In [41]:
all_regionprops_data = pd.read_csv("female_rat_regionprops.csv")

In [4]:
morphology_features = ['area', 'bbox_area', 'convex_area',
              'eccentricity', 'equivalent_diameter', 'euler_number',
              'extent', 'filled_area', 'major_axis_length',
              'minor_axis_length', 'orientation', 'perimeter', 'solidity',
              'feret_diameter_max', 'circularity', 'aspect_ratio']

In [5]:
# Define your base directory and target directories for training and testing
base_dir = "/gscratch/cheme/nlsschim/data/cross_species/rat/thresholded/tifs/female"
train_dir = "/gscratch/cheme/nlsschim/data/cross_species/rat/thresholded/tifs/female/training"
test_dir = "/gscratch/cheme/nlsschim/data/cross_species/rat/thresholded/tifs/female/testing"

# Define a list of subfolder names or patterns to look for
treatment_conditions = ["p13_female", "p70_male"]
groups = ["cblm", "cc", "cortex", "hip", "pal", "striatum", "thal"]
#groups = ["human_female"]

# Create training and testing directories if they don't exist
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

In [ ]:
image_set_path = base_dir

train_properties_df = vampire.extraction.extract_properties(img_set_path=image_set_path)

In [6]:
build_info_df = pd.DataFrame({
    'img_set_path': [image_set_path],
    'output_path': [image_set_path],
    'model_name': ['li'],
    'num_points': [50],
    'num_clusters': [5],
    'num_pc': [np.nan]
})

In [ ]:
vampire.quickstart.fit_models(build_info_df)

In [8]:
model_path = os.path.join(image_set_path, 'model_li_(50_5_25)__.pickle')
vampire_model = vampire.util.read_pickle(model_path)

In [16]:
#groups = ["human_female", "human_male",  "mice_female",  "mice_male",  "pig_female",  "rat_female",  "rat_male"]

In [7]:
from pathlib import Path
from collections import defaultdict
import random
import shutil
import tempfile
from typing import Optional

def create_stratified_kfolds(
    base_dir: str,
    groups: list[str],
    treatment_conditions: list[str],
    k: int = 5,
    file_type: str = ".tif",
    random_seed: int = 42
) -> list[list[Path]]:
    """
    Create k stratified folds from microscopy images, ensuring equal representation
    across species/sex groups and treatment conditions.

    Stratification works by finding the minimum number of images available across
    all group/condition combinations, then sampling that many from each stratum
    to guarantee balanced folds.

    Parameters
    ----------
    base_dir : str
        Root directory structured as base_dir/group/condition/*.tif
    groups : list of str
        Species/sex groups (e.g. ["human_female", "mice_male"])
    treatment_conditions : list of str
        Brain region conditions (e.g. ["cortex", "corpus_col"])
    k : int, optional
        Number of folds. Default is 5.
    file_type : str, optional
        File extension to look for. Default is ".tif".
    random_seed : int, optional
        Random seed for reproducibility. Default is 42.

    Returns
    -------
    folds : list of list of Path
        A list of k folds, where each fold is a list of file paths.
        Each fold has equal representation from all group/condition strata.

    Notes
    -----
    - Files are sampled without replacement within each stratum.
    - Strata with fewer than k files are skipped with a warning.
    - The number of files per fold per stratum = floor(stratum_size / k).
    - Total files used = n_valid_strata * k * files_per_fold_per_stratum.

    Examples
    --------
    >>> folds = create_stratified_kfolds(
    ...     base_dir="/data/cross_species/tifs/adult_wm_gm",
    ...     groups=["human_female", "mice_male", "rat_female"],
    ...     treatment_conditions=["cortex", "corpus_col"],
    ...     k=5
    ... )
    >>> print(f"Fold sizes: {[len(f) for f in folds]}")
    """
    random.seed(random_seed)
    base_path = Path(base_dir)

    # --- Collect files per stratum ---
    strata: dict[tuple[str, str], list[Path]] = defaultdict(list)

    for group in groups:
        for condition in treatment_conditions:
            stratum_path = base_path / group / condition
            if not stratum_path.exists():
                print(f"  [WARNING] Path not found, skipping: {stratum_path}")
                continue

            files = sorted(stratum_path.glob(f"*{file_type}"))
            if len(files) < k:
                print(
                    f"  [WARNING] {group}/{condition} has only {len(files)} files "
                    f"(need >= {k}). Skipping this stratum."
                )
                continue

            strata[(group, condition)] = files
            print(f"  {group}/{condition}: {len(files)} files")

    if not strata:
        raise ValueError("No valid strata found. Check base_dir and group/condition names.")

    # --- Find the minimum per-fold contribution across strata ---
    # Each stratum contributes floor(n_files / k) files total, split evenly across folds
    files_per_fold_per_stratum = min(len(files) // k for files in strata.values())
    total_per_stratum = files_per_fold_per_stratum * k

    print(f"\nFiles per fold per stratum : {files_per_fold_per_stratum}")
    print(f"Total files used per stratum: {total_per_stratum}")
    print(f"Total strata               : {len(strata)}")
    print(f"Total files across all folds: {total_per_stratum * len(strata)}")

    # --- Build k balanced folds ---
    folds: list[list[Path]] = [[] for _ in range(k)]

    for (group, condition), files in strata.items():
        # Randomly sample exactly total_per_stratum files from this stratum
        sampled = random.sample(files, total_per_stratum)

        # Distribute evenly: fold i gets sampled[i::k] (round-robin assignment)
        for fold_idx in range(k):
            fold_files = sampled[fold_idx * files_per_fold_per_stratum :
                                 (fold_idx + 1) * files_per_fold_per_stratum]
            folds[fold_idx].extend(fold_files)

    # Shuffle within each fold so files aren't ordered by stratum
    for fold in folds:
        random.shuffle(fold)

    print(f"\nFold sizes: {[len(f) for f in folds]}")
    return folds


def kfold_to_temp_dirs(
    folds: list[list[Path]],
    fold_idx: int,
    prefix: str = "vampire_fold"
) -> tuple[Path, Path, Path]:
    """
    Copy files for a single fold into temporary train/test directories.

    Parameters
    ----------
    folds : list of list of Path
        Output of create_stratified_kfolds.
    fold_idx : int
        Index of the fold to use as the test set.
    prefix : str, optional
        Prefix for the temporary directory name.

    Returns
    -------
    temp_dir : Path
        Root temporary directory (caller is responsible for cleanup).
    train_dir : Path
        Directory containing training files.
    test_dir : Path
        Directory containing test files.

    Examples
    --------
    >>> folds = create_stratified_kfolds(base_dir, groups, conditions)
    >>> for fold_idx in range(len(folds)):
    ...     temp_dir, train_dir, test_dir = kfold_to_temp_dirs(folds, fold_idx)
    ...     try:
    ...         # run vampire training/testing here
    ...         pass
    ...     finally:
    ...         shutil.rmtree(temp_dir)
    """
    temp_dir = Path(tempfile.mkdtemp(prefix=f"{prefix}{fold_idx}_"))
    train_dir = temp_dir / "train"
    test_dir = temp_dir / "test"
    train_dir.mkdir()
    test_dir.mkdir()

    test_files = folds[fold_idx]
    train_files = [f for i, fold in enumerate(folds) if i != fold_idx for f in fold]

    for f in train_files:
        shutil.copy2(f, train_dir / f.name)
    for f in test_files:
        shutil.copy2(f, test_dir / f.name)

    print(f"Fold {fold_idx}: train={len(train_files)}, test={len(test_files)}")
    return temp_dir, train_dir, test_dir

In [8]:
# Define your base directory and target directories for training and testing
base_dir = "/gscratch/cheme/nlsschim/data/cross_species/rat/thresholded/tifs/female"
train_dir = "/gscratch/cheme/nlsschim/data/cross_species/rat/thresholded/tifs/female/training"
test_dir = "/gscratch/cheme/nlsschim/data/cross_species/rat/thresholded/tifs/female/testing"

# Define a list of subfolder names or patterns to look for
treatment_conditions = ["p13_female", "p70_male"]
groups = ["cblm", "cc", "cortex", "hip", "pal", "striatum", "thal"]
#groups = ["human_female"]

# Create training and testing directories if they don't exist
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

In [ ]:
base_dir = "/gscratch/cheme/nlsschim/data/cross_species/rat/thresholded/tifs/female"

groups = ["p13_female", "p70_female"]
treatment_conditions = ["cblm", "cc", "cortex", "hip", "pal", "striatum", "thal"]

k = 2

folds = create_stratified_kfolds(
    base_dir=base_dir,
    groups=groups,
    treatment_conditions=treatment_conditions,
    k=k,
    random_seed=None
)


In [ ]:
# ── K-fold cross-validation loop ───────────────────────────────────────────────

all_predictions = []
all_centroids = []
all_models = []

import vampire

for fold_idx in range(len(folds)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1}/{k}")
    print(f"{'='*60}")

    test_files = folds[fold_idx]
    train_files = [f for j, fold in enumerate(folds) if j != fold_idx for f in fold]

    print(f"Train: {len(train_files)}, Test: {len(test_files)}")

    # Create temp directories
    temp_dir = Path(tempfile.mkdtemp(prefix=f'vampire_fold{fold_idx}_'))
    train_dir = temp_dir / "train"
    test_dir = temp_dir / "test"
    train_dir.mkdir()
    test_dir.mkdir()

    # Copy files
    for f in train_files:
        shutil.copy2(f, train_dir / f.name)
    for f in test_files:
        shutil.copy2(f, test_dir / f.name)

    try:
        # Extract features
        print("Extracting features...")
        vampire.extraction.extract_properties(str(train_dir))

        # Train model
        print("Training model...")
        build_info_df = pd.DataFrame({
            'img_set_path': [str(train_dir)],
            'output_path': [str(train_dir)],
            'model_name': [f'model_fold{fold_idx}'],
            'num_points': [50],
            'num_clusters': [5],
            'num_pc': [10]
        })
        vampire.quickstart.fit_models(build_info_df)

        # Find model file and load centroids
        model_files = [f for f in train_dir.glob("*.pickle") if "model_" in f.name]
        if not model_files:
            raise FileNotFoundError(f"No model .pickle found in {train_dir}")
        model_path = model_files[0]

        with open(model_path, 'rb') as f:
            model = pickle.load(f)

        all_models.append(model)   # model = pickle.load(open(model_path, 'rb'))

        centroids = model.centroids
        print(f"Centroids shape: {centroids.shape}")
        all_centroids.append(centroids)

        # Apply to test set
        print("Applying to test set...")
        apply_info_df = pd.DataFrame({
            'img_set_path': [str(test_dir)],
            'model_path': [str(model_path)],
            'output_path': [str(test_dir)],
            'img_set_name': [f'fold{fold_idx}']
        })
        vampire.quickstart.transform_datasets(apply_info_df)#, savefig=False)

        # Load predictions
        pred_files = list(test_dir.glob("*apply*.csv"))
        if not pred_files:
            raise FileNotFoundError(f"No predictions CSV found in {test_dir}")

        pred_df = pd.read_csv(pred_files[0])
        pred_df['fold'] = fold_idx
        pred_df['original_shape_mode'] = pred_df['cluster_id']



            # Silhouette score on the fly
        feature_cols = [f for f in morphology_features if f in pred_df.columns]
        if len(feature_cols) >= 2 and pred_df['cluster_id'].nunique() >= 2:
            X_scaled = StandardScaler().fit_transform(pred_df[feature_cols].dropna())
            clean_labels = pred_df.loc[pred_df[feature_cols].dropna().index, 'cluster_id']
            fold_sil_score = silhouette_score(X_scaled, clean_labels)
            pred_df['silhouette_fold_score'] = fold_sil_score  # store on df for later aggregation
            print(f"✓ Collected {len(pred_df)} predictions | Silhouette: {fold_sil_score:.3f}")
        else:
            print(f"✓ Collected {len(pred_df)} predictions | Silhouette: skipped (insufficient clusters/features)")



        all_predictions.append(pred_df)
        print(f"✓ Collected {len(pred_df)} predictions")

    except Exception as e:
        print(f"[ERROR] Fold {fold_idx} failed: {e}")
        raise

    finally:
        shutil.rmtree(temp_dir)

In [ ]:
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
import numpy as np
import pandas as pd

def align_fold_labels(all_centroids: list[np.ndarray], 
                      all_predictions: list[pd.DataFrame],
                      reference_fold: int = 0) -> pd.DataFrame:
    """
    Align cluster labels across folds using the Hungarian algorithm on centroid distances.

    Cluster labels from unsupervised models (e.g., VAMPIRE) are arbitrary per fold.
    This function selects one fold as the reference and remaps all other folds'
    cluster labels to best match it via optimal bipartite assignment.

    Parameters
    ----------
    all_centroids : list of np.ndarray
        List of centroid arrays, one per fold. Each array has shape
        (num_clusters, num_features).
    all_predictions : list of pd.DataFrame
        List of prediction DataFrames, one per fold. Must contain a 'cluster_id'
        column with 0-indexed integer labels.
    reference_fold : int, optional
        Index of the fold to use as the label reference. Default is 0.

    Returns
    -------
    pd.DataFrame
        Concatenated predictions with an added 'aligned_cluster_id' column
        reflecting the remapped labels.

    Notes
    -----
    - Centroid arrays must have consistent shape across folds (same num_clusters 
      and num_features). If VAMPIRE uses PCA internally, ensure num_pc is fixed.
    - The original 'cluster_id' column is preserved for traceability.
    - Uses Euclidean distance between centroids for cost matrix construction.

    Examples
    --------
    >>> aligned_df = align_fold_labels(all_centroids, all_predictions, reference_fold=0)
    >>> aligned_df.groupby(['fold', 'aligned_cluster_id']).size()
    """
    reference_centroids = all_centroids[reference_fold]
    
    aligned_dfs = []
    for fold_idx, (centroids, pred_df) in enumerate(zip(all_centroids, all_predictions)):
        
        pred_df = pred_df.copy()
        
        if fold_idx == reference_fold:
            pred_df['aligned_cluster_id'] = pred_df['cluster_id']
            aligned_dfs.append(pred_df)
            continue
        
        # Build cost matrix: distance from each fold centroid to each reference centroid
        # Shape: (num_clusters_this_fold, num_clusters_reference)
        cost_matrix = cdist(centroids, reference_centroids, metric='euclidean')
        
        # Hungarian algorithm: finds optimal 1-to-1 assignment minimising total cost
        fold_cluster_indices, reference_cluster_indices = linear_sum_assignment(cost_matrix)
        
        # Build remapping dict: fold's label -> reference's label
        label_map = {
            int(fold_idx): int(ref_idx)
            for fold_idx, ref_idx in zip(fold_cluster_indices, reference_cluster_indices)
        }
        
        print(f"Fold {fold_idx} label mapping: {label_map}")
        
        pred_df['aligned_cluster_id'] = pred_df['cluster_id'].map(label_map)
        aligned_dfs.append(pred_df)
    
    aligned_df = pd.concat(aligned_dfs, ignore_index=True)
    return aligned_df


# ── Usage ──────────────────────────────────────────────────────────────────────

aligned_predictions = align_fold_labels(all_centroids, all_predictions, reference_fold=0)

# Sanity check: distribution of aligned labels per fold
print("\nAligned cluster distribution per fold:")
print(
    aligned_predictions
    .groupby(['fold', 'aligned_cluster_id'])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pickle
from pathlib import Path


def plot_aligned_contours(all_models: list,
                          all_predictions: list[pd.DataFrame],
                          aligned_predictions: pd.DataFrame,
                          num_clusters: int = 5,
                          reference_fold: int = 0) -> plt.Figure:
    """
    Plot mean contours per aligned cluster across folds to validate label alignment.

    Each column corresponds to an aligned cluster (shape mode). Each row is a fold.
    If alignment is correct, contours in the same column should look morphologically
    similar across rows.

    Parameters
    ----------
    all_models : list
        Loaded VAMPIRE model objects, one per fold. Each must expose a
        `.contours` or `.aligned_contours` array of shape
        (n_cells, num_points, 2).
    all_predictions : list of pd.DataFrame
        Raw per-fold prediction DataFrames with 'cluster_id' column.
    aligned_predictions : pd.DataFrame
        Output of align_fold_labels(), containing 'fold' and 'aligned_cluster_id'.
    num_clusters : int
        Number of shape mode clusters.
    reference_fold : int
        Which fold to highlight with a border (the alignment reference).

    Returns
    -------
    plt.Figure
    """
    n_folds = len(all_models)
    fig, axes = plt.subplots(
        n_folds, num_clusters,
        figsize=(num_clusters * 2.5, n_folds * 2.5),
        constrained_layout=True
    )

    # Consistent color per aligned cluster across all panels
    colors = plt.cm.tab10(np.linspace(0, 1, num_clusters))

    for fold_idx, model in enumerate(all_models):
        # Pull contour coordinates — shape (n_cells, num_points, 2)
        # VAMPIRE stores these as model.contours after fitting
        contours = model.contours   # adjust attribute name if needed

        fold_mask = aligned_predictions['fold'] == fold_idx
        fold_df = aligned_predictions[fold_mask].copy()
        fold_df = fold_df.reset_index(drop=True)

        for cluster_idx in range(num_clusters):
            ax = axes[fold_idx, cluster_idx]

            cluster_mask = fold_df['aligned_cluster_id'] == cluster_idx
            cell_indices = fold_df[cluster_mask].index.tolist()

            if len(cell_indices) == 0:
                ax.text(0.5, 0.5, 'no cells', ha='center', va='center',
                        transform=ax.transAxes, fontsize=8, color='gray')
                ax.axis('off')
                continue

            # Subset contours and compute mean shape
            cluster_contours = contours[cell_indices]  # (n, num_points, 2)
            mean_contour = cluster_contours.mean(axis=0)  # (num_points, 2)

            # Plot individual contours faintly, mean contour boldly
            for c in cluster_contours[::max(1, len(cell_indices)//20)]:  # subsample max 20
                ax.plot(c[:, 0], c[:, 1], color=colors[cluster_idx], alpha=0.15, lw=0.5)

            ax.plot(mean_contour[:, 0], mean_contour[:, 1],
                    color=colors[cluster_idx], lw=2.5, label='mean')
            # Close the contour
            ax.plot([mean_contour[-1, 0], mean_contour[0, 0]],
                    [mean_contour[-1, 1], mean_contour[0, 1]],
                    color=colors[cluster_idx], lw=2.5)

            ax.set_aspect('equal')
            ax.axis('off')
            n_cells = cluster_mask.sum()
            ax.set_title(f'n={n_cells}', fontsize=8)

            # Highlight reference fold
            if fold_idx == reference_fold:
                for spine in ax.spines.values():
                    spine.set_visible(True)
                    spine.set_edgecolor('black')
                    spine.set_linewidth(2)

        axes[fold_idx, 0].set_ylabel(f'Fold {fold_idx}', fontsize=10, rotation=90)

    # Column headers
    for cluster_idx in range(num_clusters):
        axes[0, cluster_idx].set_title(
            f'Shape mode {cluster_idx}\n'
            f'(n={( aligned_predictions["aligned_cluster_id"] == cluster_idx).sum()})',
            fontsize=9
        )

    fig.suptitle(
        'Aligned contours per shape mode across folds\n'
        'Columns should look morphologically consistent if alignment is correct',
        fontsize=12, y=1.02
    )

    return fig


# ── Usage ─────────────────────────────────────────────────────────────────────

# You need to save the model objects during your k-fold loop.
# Add this inside your loop after loading the model:
#
#   all_models.append(model)   # model = pickle.load(open(model_path, 'rb'))

fig = plot_aligned_contours(
    all_models=all_models,
    all_predictions=all_predictions,
    aligned_predictions=aligned_predictions,
    num_clusters=5,
    reference_fold=0
)
fig.savefig('aligned_contour_validation.png', dpi=150, bbox_inches='tight')
plt.show()